Project: /data-manager/api/_project.yaml
Book: /data-manager/api/_book.yaml

<style>
  devsite-code .tfo-notebook-code-cell-output {
    max-height: 300px;
    overflow: auto;
    background: rgba(255, 247, 237, 1);  /* light orange bg */
  }
  
  devsite-code .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
    background: rgba(255, 247, 237, .7);
  }
  
  devsite-code[dark-code] .tfo-notebook-code-cell-output {
    background: rgba(64, 78, 103, 1);  /* dark mode slate */
  }
  
  devsite-code[dark-code] .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
    background: rgba(64, 78, 103, .7);
  }
  
  .devsite-table-wrapper .tfo-notebook-buttons {
    display: inline-block;
    margin-left: 3px;
    width: auto;
    border: 0;
  }
  
  .tfo-notebook-buttons tr {
    background: 0;
    border: 0;
  }
  
  .tfo-notebook-buttons td {
    padding-left: 0;
    padding-right: 20px;
    border: 0;
  }
  
  .tfo-notebook-buttons {
    --tfo-notebook-buttons-box-shadow: 0 1px 2px 0 rgba(60, 64, 67, .3), 0 1px 3px 1px rgba(60, 64, 67, .15);
  }
  
  .tfo-notebook-buttons a,
  .tfo-notebook-buttons :link,
  .tfo-notebook-buttons :visited {
    border-radius: 8px;
    box-shadow: var(--tfo-notebook-buttons-box-shadow);
    color: #202124;
    padding: 12px 24px;
    transition: box-shadow 0.2s;
    text-decoration: none;
    display: flex;
    align-items: center;
  }
  
  .tfo-notebook-buttons a:hover,
  .tfo-notebook-buttons a:focus {
    box-shadow: 0 2px 6px 2px rgba(60, 64, 67, 0.15);
    text-decoration: none;
  }
  
  .tfo-notebook-buttons td > a > img {
    margin-right: 8px;
    width: 32px;
    height: 32px;
  }
  </style>

In [ ]:
# @markdown #### Copyright 2026 Google LLC
# @markdown ##### Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Advertiser Flow

  <table class="tfo-notebook-buttons nocontent" align="left">
    <td>
      <a target="_blank" href="https://colab.research.google.com/github/googleads/data-manager-python/blob/main/notebooks/audience_e2e_advertiser_flow.ipynb">
      <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
      Run in Google Colab</a>
    </td>
    <td>
      <a target="_blank" href="https://github.com/googleads/data-manager-python/blob/main/notebooks/audience_e2e_advertiser_flow.ipynb">
      <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
      View source on GitHub</a>
    </td>
  </table>

## Objective
This notebook provides the workflow for an Advertiser operating on their own Google Ads account.
It can authenticate using either user credentials or a service account.

## Prerequisites

A Google Account with the **Standard**
[access level](https://support.google.com/google-ads/answer/9978556) in your Google Ads account or
its parent Google Ads manager account. Provide credentials to act as this user using one of the
following options in Step 2:

- A client ID, client secret, and refresh token.

  Use this option if you've already gone through the process of generating user credentials.

- The JSON for a **Desktop app** OAuth client.

  [Follow these instructions](https://developers.google.com/data-manager/api/devguides/quickstart/set-up-access#user-account) if you don't have this JSON file. You'll use the JSON file you download for the prompt in **Step 1**.

- The email address of a [service account](//cloud.google.com/docs/authentication#service-accounts).

   [Follow these instructions](https://developers.google.com/data-manager/api/devguides/quickstart/set-up-access#service-account) if you don't have a service account.

## Instructions
1. Make a copy of this Colab.
2. Fill in the prompts in **Step 1**.
3. Run your copy of the Colab.


### Step 0. Install and Import Packages

In [ ]:
!pip install --upgrade google-ads-datamanager google-auth-oauthlib

In [ ]:
import datetime
import getpass
from google.ads import datamanager_v1
from google.api_core import exceptions
import google.auth
from google.colab import files
from google.oauth2.credentials import Credentials
from google.protobuf.json_format import MessageToJson

### Step 1. Setup and Configuration

In [ ]:
authentication_method = "OAuth Client Credentials"  # @param ["OAuth Client Credentials", "Service Account"]

# @markdown *Check this if you already have a Client ID, Client Secret, and Refresh Token to bypass the gcloud setup:*
use_existing_credentials = False  # @param {type:"boolean"}

# @markdown ### Account IDs
# @markdown **ADVERTISERS:**
# @markdown - **[required]** `operating_account_id`: Your advertiser account
# @markdown - (*optional*) `login_account_id`: Don't set this if the Google Account of your credentials is a user in the advertiser account. If instead the Google Account has access to a Google Ads manager account with the advertiser account as a child, set this to the customer ID of the Google Ads manager account.
operating_account_id = ""  # @param {type:"string"}
login_account_id = ""  # @param {type:"string"}

# @markdown ### Service Account Details (If applicable)
service_account_email = ""  # @param {type:"string"}

# Clean up account IDs by removing hyphens.
operating_account_id = operating_account_id.replace("-", "")
login_account_id = login_account_id.replace("-", "")

GOOGLE_ADS = "GOOGLE_ADS"
CONSENT_GRANTED = "CONSENT_GRANTED"

### Step 2. Initialize Client

In [ ]:
class DataManagerSDK:
    def __init__(self, creds):
        self.user_list_service = datamanager_v1.UserListServiceClient(credentials=creds)
        self.ingestion_service = datamanager_v1.IngestionServiceClient(credentials=creds)

def prompt_for_desktop_client_json():

    # Prompts the user to upload the JSON file
    print("Upload the client JSON file for your Google Cloud Desktop OAuth client:")
    uploaded = files.upload()
    desktop_app_json_file = "/tmp/oauth_client.json"

    if len(uploaded) != 1:
        raise ValueError("Please upload exactly one file.")

    filename = list(uploaded.keys())[0]
    content = uploaded[filename]

    # Saves the uploaded file.
    with open(desktop_app_json_file, "wb") as f:
        f.write(content)
    # Returns the file path and name.
    return desktop_app_json_file

def initialize_client():
    data_manager_scope = "https://www.googleapis.com/auth/datamanager";
    creds = None

    if authentication_method == "OAuth Client Credentials":
        if use_existing_credentials:
            print("Please enter your OAuth credentials:")
            client_id = input("Client ID: ").strip()
            client_secret = getpass.getpass("Client Secret (input will be hidden): ").strip()
            refresh_token = getpass.getpass("Refresh Token (input will be hidden): ").strip()

            creds = Credentials(
                token=None,
                client_id=client_id,
                client_secret=client_secret,
                refresh_token=refresh_token,
                token_uri="https://oauth2.googleapis.com/token",
                scopes=[data_manager_scope]
            )
        else:
            print("Authenticating with OAuth via gcloud...")
            client_json_file = prompt_for_desktop_client_json()
            auth_command = (
                f"gcloud auth application-default login "
                f"--client-id-file='{client_json_file}' "
                f"--scopes={data_manager_scope},https://www.googleapis.com/auth/cloud-platform "
                f"--no-browser"
            )
            print("Running gcloud auth command...")
            get_ipython().system(auth_command)
            print("gcloud auth process finished")
            creds, project = google.auth.default(scopes=[data_manager_scope])
    else:
        print("Authenticating with Service Account...")
        auth_command = (
            f"gcloud auth application-default login "
            f"--impersonate-service-account={service_account_email} "
            f"--scopes={data_manager_scope},https://www.googleapis.com/auth/cloud-platform "
            f"--no-browser"
        )
        print("Running gcloud auth command...")
        get_ipython().system(auth_command)
        print("gcloud auth process finished")
        creds, project = google.auth.default(scopes=[data_manager_scope])

    return DataManagerSDK(creds)

sdk = initialize_client()

### Step 3. Create User List

In [ ]:
print(f"Creating User List in {operating_account_id}...\n")

parent_userlist = f"accountTypes/GOOGLE_ADS/accounts/{operating_account_id}"

user_list_data = datamanager_v1.UserList(
    display_name=f"Python SDK Audience - {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    ingested_user_list_info=datamanager_v1.IngestedUserListInfo(
        upload_key_types=["CONTACT_ID"]
    ),
)

headers = []
if login_account_id:
    headers.append(
        (
            "login-account",
            f"accountTypes/GOOGLE_ADS/accounts/{login_account_id}",
        )
    )

destination_id = None
try:
    ulist_res = sdk.user_list_service.create_user_list(
        parent=parent_userlist, user_list=user_list_data, metadata=headers
    )

    destination_id = ulist_res.id

    print(f"User List Created!")
    print(f"Destination ID: {destination_id}")
    print(f"Resource Name: {ulist_res.name}")
    print(f"Display Name: {ulist_res.display_name}")

except exceptions.PermissionDenied as e:
    print(f"User list creation failed due to permission denied error: {e}")
    if authentication_method == "OAuth Client Credentials":
        print(
            "Review the 'Prerequisites' section and verify that the Google Account of the user "
            "credentials has the required access level in the advertiser account."
        )
    else:
        print(
            "Review the 'Prerequisites' section and verify that the service account has the "
            "required access level in the advertiser account."
        )
except Exception as e:
    print(f"Failed to create User List: {e}")

print(
    "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/accountTypes.accounts.userLists/create?apix=true"
)
print(f"\nparent: {parent_userlist}")

print("\n--- Request JSON Payload ---")
print(MessageToJson(user_list_data._pb))
print("----------------------------\n")

### Step 4. Ingest Data

In [ ]:
ingestion_request_id = None

if not destination_id:
    print(
        "Error: destination_id is not set. Cannot ingest data. Please ensure the User List was created successfully in the previous step."
    )
else:
    print(f"Ingesting sample data to User List {destination_id}...")

    destination_obj = datamanager_v1.Destination(
        operating_account=datamanager_v1.ProductAccount(
            account_type=GOOGLE_ADS, account_id=operating_account_id
        ),
        product_destination_id=str(destination_id),
    )

    if login_account_id:
        destination_obj.login_account = datamanager_v1.ProductAccount(
            account_type=GOOGLE_ADS, account_id=login_account_id
        )

    ingest_payload = datamanager_v1.IngestAudienceMembersRequest(
        consent=datamanager_v1.Consent(
            ad_user_data=CONSENT_GRANTED, ad_personalization=CONSENT_GRANTED
        ),
        encoding=datamanager_v1.Encoding.HEX,
        terms_of_service=datamanager_v1.TermsOfService(
            customer_match_terms_of_service_status="ACCEPTED"
        ),
        validate_only=False,
        audience_members=[
            datamanager_v1.AudienceMember(
                user_data=datamanager_v1.UserData(
                    user_identifiers=[
                        datamanager_v1.UserIdentifier(
                            email_address="223EBDA6F6889B1494551BA902D9D381DAF2F642BAE055888E96343D53E9F9C4"
                        ),
                        datamanager_v1.UserIdentifier(
                            email_address="F1FCDE379F31F4D446B76EE8F34860ECA2288ADC6B6D6C0FDC56D9EEE75A2FA5"
                        ),
                    ]
                )
            )
        ],
        destinations=[destination_obj],
    )

    try:
        ingest_res = sdk.ingestion_service.ingest_audience_members(
            request=ingest_payload
        )

        ingestion_request_id = ingest_res.request_id
        print(f"Ingestion Submitted!")
        print(f"Request ID: {ingestion_request_id}")

    except Exception as e:
        print(f"Failed to ingest data: {e}")

    print(
        "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/audienceMembers/ingest?apix=true"
    )
    print("\n--- Request JSON Payload ---")
    print(MessageToJson(ingest_payload._pb))
    print("----------------------------\n")

### Step 5. Check Status

In [ ]:
if not ingestion_request_id:
    print("Status check skipped because ingestion request ID is not set.")
else:
    print(f"5. Checking status for Request ID: {ingestion_request_id}...")

    # Note: Headers aren't needed for this request.
    status_req = datamanager_v1.RetrieveRequestStatusRequest(
        request_id=ingestion_request_id
    )

    try:
        status_res = sdk.ingestion_service.retrieve_request_status(
            request=status_req
        )

        print(f"Successfully retrieved status!")

        for dest_status in status_res.request_status_per_destination:
            dest_id = dest_status.destination.product_destination_id
            current_status = dest_status.request_status.name

            print(f"Status for destination {dest_id}: {current_status}")

    except Exception as e:
        print(f"Failed to retrieve status: {e}")

    print(
        "\nAPIx link: https://developers.google.com/data-manager/api/reference/rest/v1/requestStatus/retrieve?apix=true"
    )
    print("\n--- Request JSON Payload ---")
    print(MessageToJson(status_req._pb))
    print("----------------------------")